# Final Assignment4 - Green Patent Detection: Advanced Agentic Workflow with QLoRA (Option: Mistral-7B and LangGraph)
> Suchanya Baiyam Business Data Science AAU
 - Build a state-of-the-art data labeling pipeline
 - Fine-tune an LLM using QLoRA to understand patent language
 - Integrate that fine-tuned model into a Multi-Agent System (MAS) to debate and label complex claims


Part C STEP02: Multi-Agent System (MAS)
- Serve newly fine-tuned QLoRA model as the "brain" for at least one of your agents using a framework as CrewAI

Required Agents to label the 100 high-risk claims:
- Agent 1 (The Advocate): Argues for the green classification.
- Agent 2 (The Skeptic): Argues against it (identifying greenwashing).
- Agent 3 (The Judge): Weighs the arguments and produces the final JSON label + rationale.

Load Adapted Model from Colab 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
adapter_path = "/content/drive/MyDrive/green_patent_lora"

STEP01: Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

STEP02: Load dataset

In [ ]:
df = pd.read_parquet("patent_50k_green.parquet")
df.shape

(50000, 4)

In [ ]:
df.head()

,doc_id,text,is_green_silver,split
402589,9669654,1. A paint-bucket mesh with a roller cleaning ...,0,train_silver
89394,8984304,1. A method for reducing power consumption in ...,1,train_silver
181025,9234489,1. A method for operating a temperature-limiti...,1,train_silver
25822,9001929,1. A method for a transmitter to transmit data...,0,train_silver
197203,8499387,"1. A treatment couch having a lying surface, a...",0,train_silver


In [ ]:
high_risk_100 = pd.read_csv("hitl_green_100.csv")
high_risk_100.shape

(100, 9)

STEPS LOCALLY RUN MAS: run model locally, create graph state, agent nodes, build langgraph, run pipeline with 100 claims, save output in csv

In [ ]:
results_df = pd.read_csv("mas_results.csv")

Part D:Targeted Human Review & Final Integration

From the instruction that following:
- Exception-Based HITL: You no longer need to review all 100 claims! You will only step in as the "Human-in-the-Loop" for claims where the Advocate and Skeptic reach a deadlock or if confidence is low. For these disagreements, read the AI rationale and provide your human judgment to create the final is_green_gold label. Accept the Judge's decision for the rest.
- Disagreement Reporting: In your report, detail how many claims out of the 100 required human intervention (e.g., "The agents disagreed on 14 out of 100 claims").

STEP01: Find claims that LLM uncertain

In [ ]:
#code structured by claude but the keywords define by me
uncertain_keywords = ["unclear", "uncertain", "debateable", "unsure", "however", "uncertainly"]
hitl_needed = results_df["judge_rationale"].str.lower().str.contains("|".join(uncertain_keywords), na=False)
hitl_needed = results_df[
    results_df["judge_rationale"].str.lower().str.contains(
        "|".join(uncertain_keywords), na=False
    )
]

print(f"Claims that needs human review: {len(hitl_needed)} from 100")
hitl_needed[["doc_id", "advocate_argument", "skeptic_argument", "is_green_tech", "judge_rationale"]]

Claims that needs human review: 2 from 100


,doc_id,advocate_argument,skeptic_argument,is_green_tech,judge_rationale
7,9206804,This patent for a compressor for vehicles coul...,This claim may not be environmentally friendly...,1,While the compressor may contribute to energy ...
23,9122206,This patent for a liquid toner composition cou...,While the claim does not inherently indicate e...,1,While the claim does not guarantee environment...


In [ ]:
#code assisted by cluade
for _, row in hitl_needed.iterrows():
    print(f"=== doc_id: {row['doc_id']} ===")
    print(f"TEXT:\n{row['text']}\n")
    print(f"ADVOCATE:\n{row['advocate_argument']}\n")
    print(f"SKEPTIC:\n{row['skeptic_argument']}\n")
    print(f"JUDGE RATIONALE:\n{row['judge_rationale']}")
    print(f"CURRENT is_green_tech: {row['is_green_tech']}")
    print("="*60 + "\n")

=== doc_id: 9206804 ===
TEXT:
1. A compressor for vehicle including a housing, the compressor comprising: a compression portion located in the housing; a rotary shaft connected with the compression portion at one end; a motor driving the compression portion through the rotary shaft; a motor accommodation chamber accommodating the motor; a suction port formed in the housing and sucking fluid into the motor accommodation chamber; and a first shaft portion and a second shaft portion formed at both ends of the rotary shaft; wherein the first shaft portion is formed between the compression portion and the motor, and is supported by a first bearing, wherein the second shaft portion is formed between the motor and the housing, and is supported by a second bearing, wherein a coil spring is wound around the second shaft portion so as to rotate with the second shaft portion, wherein the rotary shaft forms a spring supporting portion and the diameter of which is greater than that of the second sh

STEP02: Make is_tech_gold by justify the claims that makes judge LLM giving uncertain results

In [ ]:
results_df["is_green_gold"] = results_df["is_green_tech"]
results_df.loc[results_df["doc_id"] == 9206804, "is_green_gold"] = 0
results_df.loc[results_df["doc_id"] == 9122206, "is_green_gold"] = 0
print(results_df[["doc_id", "is_green_tech", "is_green_gold"]].head(10))

    doc_id  is_green_tech  is_green_gold
0  9315787              0              0
1  9334087              1              1
2  9686855              0              0
3  9551706              0              0
4  9745777              0              0
5  9237703              0              0
6  9273641              0              0
7  9206804              1              0
8  8466241              0              0
9  8581146              0              0


STEP03: Save Result is_green_gold

In [ ]:
results_df.to_csv("mas_results_gold.csv", index=False)

STEP04: Fine Tuning
STEP04.1: Load libraries
#reuse code from assignment 3 originally suggested by chatGPT and claude

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import evaluate
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

STEP04.02 Load Dataset & Merge Gold

In [ ]:
df = pd.read_parquet("patent_50k_green.parquet")

In [ ]:
gold = pd.read_csv("mas_results_gold.csv")[["doc_id", "is_green_gold"]]

In [ ]:
gold

,doc_id,is_green_gold
0,9315787,0
1,9334087,1
2,9686855,0
3,9551706,0
4,9745777,0
...,...,...
95,9474158,0
96,9276477,1
97,8624822,0
98,9814135,0


In [ ]:
df = df.merge(gold, on="doc_id", how="left")
df["is_green_final"] = df["is_green_gold"].fillna(df["is_green_silver"]).astype(int)
print(df.shape)

(50000, 6)


In [ ]:
df

,doc_id,text,is_green_silver,split,is_green_gold,is_green_final
0,9669654,1. A paint-bucket mesh with a roller cleaning ...,0,train_silver,NaN,0
1,8984304,1. A method for reducing power consumption in ...,1,train_silver,NaN,1
2,9234489,1. A method for operating a temperature-limiti...,1,train_silver,NaN,1
3,9001929,1. A method for a transmitter to transmit data...,0,train_silver,NaN,0
4,8499387,"1. A treatment couch having a lying surface, a...",0,train_silver,NaN,0
...,...,...,...,...,...,...
49995,9459891,1. A system comprising: a processor having a s...,0,eval_silver,NaN,0
49996,8676239,1. A wireless communication system comprising ...,1,eval_silver,NaN,1
49997,9365529,"1. A compound of formula (I), a salt thereof, ...",1,eval_silver,NaN,1
49998,9072125,1. An apparatus comprising: a controller to pr...,1,eval_silver,NaN,1


In [ ]:
print(df.columns.tolist())
print(df["is_green_final"].value_counts())

['doc_id', 'text', 'is_green_silver', 'split', 'is_green_gold', 'is_green_final']
is_green_final
0    25030
1    24970
Name: count, dtype: int64


STEP04.03: train/eval split

In [ ]:
train_df = df[df["split"] == "train_silver"].copy()
eval_df  = df[df["split"] == "eval_silver"].copy()

print(train_df.shape, eval_df.shape)

(30000, 6) (10000, 6)


In [ ]:
df["split"].value_counts()

,count
split,
train_silver,30000
pool_unlabeled,10000
eval_silver,10000


In [ ]:
gold_df = df[df["is_green_gold"].notnull()].copy()
print(gold_df.shape)

(100, 6)


STEP04.4: Conver to HF Dataset

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

eval_dataset = Dataset.from_pandas(
    eval_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

gold_dataset = Dataset.from_pandas(
    gold_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

STEP04.05: Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "Ailee52/PatentSBERTa_finetuned_green"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset  = eval_dataset.map(tokenize_function, batched=True)
gold_dataset  = gold_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/365 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

STEP04.06: Remove text & set torch

In [ ]:
train_dataset = train_dataset.remove_columns(["text"])
eval_dataset  = eval_dataset.remove_columns(["text"])
gold_dataset  = gold_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
eval_dataset.set_format("torch")
gold_dataset.set_format("torch")

STEP04.07: Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

STEP04.08 Metric(F1)

In [ ]:
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "f1": metric.compute(predictions=predictions, references=labels)["f1"]
    }

STEP04.09: Training Arguments

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./results_a4",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


STEP04.10: Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

STEP04.11: Train

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.386883,0.461010,0.794118
2,0.340263,0.467879,0.810360
3,0.275838,0.500622,0.809172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5625, training_loss=0.33218038397894967, metrics={'train_runtime': 5090.7063, 'train_samples_per_second': 17.679, 'train_steps_per_second': 1.105, 'total_flos': 1.18399974912e+16, 'train_loss': 0.33218038397894967, 'epoch': 3.0})

STEP04.12: Evaluate

In [ ]:
print("Eval Silver:")
print(trainer.evaluate(eval_dataset))

Eval Silver:


{'eval_loss': 0.500622034072876, 'eval_f1': 0.809172325558508, 'eval_runtime': 151.2183, 'eval_samples_per_second': 66.13, 'eval_steps_per_second': 4.133, 'epoch': 3.0}


In [ ]:
print("Gold 100:")
print(trainer.evaluate(gold_dataset))

Gold 100:


{'eval_loss': 1.0745975971221924, 'eval_f1': 0.2727272727272727, 'eval_runtime': 1.5177, 'eval_samples_per_second': 65.891, 'eval_steps_per_second': 4.612, 'epoch': 3.0}


STEP04.13: Save Model

In [ ]:
trainer.save_model("PatentSBERTa_finetuned_green_advanced")
tokenizer.save_pretrained("PatentSBERTa_finetuned_green_advanced")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('PatentSBERTa_finetuned_green_advanced/tokenizer_config.json',
 'PatentSBERTa_finetuned_green_advanced/tokenizer.json')

Part E: Comparative Analysis (on eval_silver)
- Baseline --> F1 Score of 0.77
- Assignment 2 Model --> F1 Score of 0.8009
- Assignment 3 Model --> F1 Score of 0.8066
- Final Assignment Model --> F1 Score of 0.8091

# Upload model to Hugging Face


*   https://huggingface.co/Ailee52/PatentSBERTa_finetuned_green_advanced
*   https://huggingface.co/datasets/Ailee52/PatentSBERTa_finetuned_green_final_advanced-dataset

